### order_items - Product and item level Analysis
- This table contains details of each product within an order, including price and shipping cost. 
- Even on its own, it provides rich commercial insights. 

In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.order_items;

Some analysis that can be done on the order items table. 
### Average Product price and Freight cost
- Calculate the average price and freight_value per order item
- Analyze cost distribution across different product types. 
### Freight ration analysis
- Compute freight_value / price to identify products wiht disproportionately high shipping costs. 
### Total item value (per Order Line)
- Use total_item_value = price + freight_value to measure the average transactional value per product
### Time-based shipping trends
- use shipping_limit_date to build daily, monthly or yearly shipping trend visualizations. 


## order_payments - Financial Transactions and Payment Behavior
- This dataset records payment methods, installment counts, and payment amounts 

In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.order_payments;

Databricks visualization. Run in Databricks to view.

Analysis that can be done on the payments table
### Payment method distribution
- Anlayze the share of card, boleto, voucher or debit payments.
- Visualize a pie or bar chart to show popular payment preferences
### Average number of installments
- Measure customer's average installment count to understand payment flexibility and behavior
### Average installment value
- Use the previously calculated payment_per_installment metric to examine credit card usage patterns. 
### Time-based payment analysis
- Examine payments by week or month to uncover seasonality in purchasing behavior. 
### Total and average payment volume
- SUM(payment_value) --> total transaction volume
- AVG(payment_value) --> average order payment amount. 

## order_reviews - Customer Feedback and Sentiment Analysis
This table contains customer ratings and textual reviews, enabling analysis of satisfaction and quality trends. 


In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.order_reviews;

Analysis that can be done on order reviews
### Average Review Score distribution
- calculate the percentage of each score (1-5) to assess overall satisfaction
### Sentiment distribution (Positive Vs Negative)
- Classify 4-5 stars as 'positive' and 1-2 stars as 'negative' to analyze sentiment balance
### Missing comment ratio
- Determine how many customers gave a rating without writing a comment - a proxy for engagement quality. 
### Title-only Reviews
- Identify reviews containing only a title to measure short-form feedback frequency. 

### geolocation - Geographic Data Analysis
This table stores coordinates and zip prefixes, making it ideal for regional coverage and quality assessment. 

In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.geolocation;

### Zip code density analysis
- count the number of records per zip_prefix to identify geographic concentration areas
### Geographic centroid validation
- use average lat and lng to determine the representative geographic center for each region. 
- This helps verify data consistency and detect outliers. 


### _reviews_bad_rows - Data Quality and Error Audit
This quarantine table contains invalid or malformed records excluded from the main dataset. 

In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.reviews_bad_rows;

### Error Row Ratio
- measure the proportion of invalid rows relative to the total dataset - a key data quality metric
### Error type disribution
- idnetify which columns most frequently cause errors (e.g, incorect date or text types).
### Missign field analysis
- Determine which columns have the highest proportion of null or empty values. 
### Fixalbe error ratio
- Measure how many errors could be corrected automatically (e.g., format or null-related)

### orders - orders lifecycle and performance analysis
This dataset caputre the entire order process - from purchase to delivery. it's ideal for time, completeness and performance based analytics. 

In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.orders;

### Order status breakdown
- group by order_status to measure completed, cancelled and pending order ratios
- pie or bar chart --> visualize operational success rates. 
### time-of-day purchase analysis
- use purchase_hour or purchase_day_of_week to identify when cusomters shop most frequently
### weekend purchase behavior
- analyze is_weekend_purchase to compare weekday vs weekend order patterns. 
### timeflow integrity check
- use is_invalid_timeflow to detect temporal inconsistencies (e.g, delivery before approval)
### missing data completeness analysis
- Analyze is_missing_customer_date, is_missing_carrier_date and is_missing_approved to see where data is most often incomplete.
### Montly order trends (seasonality)
- use purhcase_date or approved_date to buld time-series vidlizations of order volumes.
### estimated delivery benchmark
- compare actuals vs estimated delivery dates to evaluate forecast accuracy. 

## Unified Order Gold Analytics

From the Order items table, we'll calculate each order's total value, order, underscore value, and number of products item underscore count.

From the order payments table, we'll extract the payment type, total payment amount, and average number of installments for each order.

From the Order Reviews table, we'll obtain each orders average customer score from the average review score.

And finally, we'll use the orders table as the base and join all these summary tables on order ID using left join.

So as a result, the Order Performance table will be a comprehensive gold table that combines each orders financial, operational and satisfaction data in a single row.

In [0]:
%sql
CREATE OR REPLACE TABLE course_training_catalog.gold_olist.order_performance AS
WITH
item_summary AS (
  SELECT
    order_id,
    ROUND(SUM(price + freight_value), 2) AS order_value,
    COUNT(*) AS item_count
  FROM
    course_training_catalog.silver_olist.order_items
  GROUP BY
    order_id
),
payment_summary AS (
  SELECT
    order_id,
    FIRST(payment_type) AS payment_type,
    ROUND(SUM(payment_value), 2) total_payment,
    AVG(NULLIF(payment_installments, 0)) AS avg_installments
    
  FROM course_training_catalog.silver_olist.order_payments
  GROUP BY order_id
), 
reviews_summary AS (
  SELECT
    order_id,
    ROUND(AVG(review_Score), 2) AS avg_review_score,
    COUNT(*) AS review_count
  FROM
    course_training_catalog.silver_olist.order_reviews
  GROUP BY
    order_id
)
SELECT
  o.order_id,
  DATE(o.order_purchase_timestamp) AS purchase_date,
  o.order_status_group,
  o.delivery_days,
  o.delay_days,
  i.order_value,
  i.item_count, 
  p.payment_type,
  p.total_payment,
  ROUND(p.avg_installments, 2) AS avg_installments,
  r.avg_review_score
FROM course_training_catalog.silver_olist.orders AS o
LEFT JOIN item_summary AS i ON o.order_id = i.order_id
LEFT JOIN payment_summary AS p ON o.order_id = p.order_id
LEFT JOIN reviews_summary AS r ON o.order_id = r.order_id


In [0]:
%sql
SELECT * FROM course_training_catalog.gold_olist.order_performance;

In [0]:
%sql
SELECT
  ROUND(AVG(order_value), 2) AS avg_order_value
FROM course_training_catalog.gold_olist.order_performance;


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS course_training_catalog.analytics_fn; 

In [0]:
%sql
-- Defining a scalar function
CREATE OR REPLACE FUNCTION course_training_catalog.analytics_fn.avg_order_value()
RETURNS DOUBLE
RETURN (
  SELECT
    ROUND(AVG(order_value), 2) AS avg_order_value
  FROM course_training_catalog.gold_olist.order_performance
);

In [0]:
%sql
SELECT course_training_catalog.analytics_fn.avg_order_value();

In [0]:
# writing the abouve function in python
from pyspark.sql import functions as F
def avg_order_value2():
    df = spark.table("course_training_catalog.gold_olist.order_performance")

    result = (
        df.agg(F/round(F.avg("order_value"), 2).alias("avg_order_value"))
        .collect()[0]["avg_order_value"]
    )
    return result

So the functions under Unity Catalog are only those created with a create function command in SQL.

These are persistent functions recorded in the system's metadata.So functions defined in Python using def or spark register not going to be written to Databricks metadata,

which is why they don't appear in the catalog and disappear when the session ends.

So in contrast, SQL functions are permanent stored in catalog visible to all users.

UDF (user defined functions) are used in SQL. thanks to UDF we can eliminate code repetitions. functions can be shared via untiy catalog which allows for centralized management of function versioning, sharing and access permissions. And this ensures that the data team can safely use the same calculation across different reports. 

Well, we'll create a parameterized table function inside Databricks Unity catalog and take input values from the user.

For example payment type min order value max delivery days.

The function will insert these parameters into the query, and it'll return a filtered and summarized table as the result.

In [0]:
%sql
CREATE OR REPLACE FUNCTION course_training_catalog.analytics_fn.order_peformance_summary(
  payment_filter STRING,
  min_order_value DOUBLE, 
  max_delivery_days INT
)
RETURNS TABLE (
  payment_type STRING,
  order_count BIGINT,
  avg_order_value DOUBLE,
  avg_review DOUBLE,
  avg_delivery_days DOUBLE
)
RETURN
SELECT
  payment_type,
  COUNT(*) AS order_count,
  ROUND(AVG(order_value), 2) AS avg_order_value,
  ROUND(AVG(avg_review_score), 2) AS avg_review,
  ROUND(AVG(delivery_days), 2) AS avg_delivery_days
FROM course_training_catalog.gold_olist.order_performance
WHERE
  (payment_filter IS NULL OR payment_type = payment_filter)
  AND (min_order_value IS NULL OR order_value >= min_order_value)
  AND (max_delivery_days IS NULL OR delivery_days <= max_delivery_days)
GROUP BY
  payment_type
ORDER BY order_count DESC;

In [0]:
%sql
SELECT * FROM course_training_catalog.analytics_fn.order_peformance_summary(NULL, NULL, NULL);

In [0]:
%sql
SELECT * FROM course_training_catalog.analytics_fn.order_peformance_summary("credit_card", 100, 5);

Analyze relationship between delivery days ad customer satisfaction


In [0]:
%sql
SELECT
  CASE 
    WHEN delivery_days <= 3 THEN "0-3 days"
    WHEN delivery_days <= 7 THEN "4-7 days"
    WHEN delivery_days <= 14 THEN "8-14 days"
    ELSE "15+ days"
  END AS delivery_speed,
  COUNT(*) As orders,
  ROUND(AVG(order_value),2) AS avg_value,
  ROUND(AVG(avg_review_score), 2) AS avg_review
FROM course_training_catalog.gold_olist.order_performance
WHERE avg_review_score IS NOT NULL
GROUP BY 1 -- grouped by delivery speed
ORDER BY orders DESC;
